# 2-Stage SFT Training: Reasoning + Format

**Strategy:**
- **Stage 1**: 15K mixed data (compact rules + full CoT + answer-only) → learn reasoning patterns
- **Stage 2**: 600 answer-only samples at low LR → polish `\boxed{}` output format

**Critical**: User prompt suffix matches official evaluation **EXACTLY**.


In [1]:
import subprocess, sys, os, glob

# --- Step 1: Install trl + datasets from our offline packages ---
# Use --no-deps to avoid dependency resolution issues in offline mode
offline_dirs = [
    "/kaggle/input/sft-offline-packages",
    "/kaggle/input/datasets/hastws/sft-offline-packages",
]
offline_dir = None
for d in offline_dirs:
    if os.path.isdir(d):
        whl_files = [f for f in os.listdir(d) if f.endswith('.whl')]
        if whl_files:
            offline_dir = d
            print(f"Found offline packages at: {d} ({len(whl_files)} wheels)")
            break

if offline_dir:
    # Install ALL wheels with --no-deps (system packages satisfy dependencies)
    whls = sorted(glob.glob(os.path.join(offline_dir, "*.whl")))
    print(f"Installing {len(whls)} wheels with --no-deps...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-deps"] + whls,
        capture_output=True, text=True, timeout=180
    )
    if result.returncode == 0:
        print("Installed all offline wheels")
    else:
        print(f"Install error: {result.stderr[-500:]}")
        # Try one by one
        for whl in whls:
            r = subprocess.run(
                [sys.executable, "-m", "pip", "install", "-q", "--no-deps", whl],
                capture_output=True, text=True, timeout=60
            )
            name = os.path.basename(whl).split('-')[0]
            if r.returncode == 0:
                print(f"  OK: {name}")
            else:
                print(f"  FAIL: {name}: {r.stderr[-100:]}")
else:
    print("No offline packages found!")
    if os.path.isdir("/kaggle/input"):
        for item in sorted(os.listdir("/kaggle/input")):
            print(f"  /kaggle/input/{item}/")

# Verify critical packages
for pkg in ["trl", "datasets"]:
    try:
        mod = __import__(pkg)
        print(f"  {pkg}={getattr(mod, '__version__', '?')}")
    except ImportError as e:
        raise RuntimeError(f"FATAL: {pkg} not importable after install: {e}")

# --- Step 2: flash_attn from dennisfong ---
try:
    import flash_attn
    print(f"flash_attn={flash_attn.__version__}")
except ImportError:
    fa_whls = glob.glob("/kaggle/input/**/*flash_attn*.whl", recursive=True)
    if fa_whls:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", fa_whls[0]],
                       capture_output=True, text=True, timeout=120)
        print(f"Installed flash_attn from {os.path.basename(fa_whls[0])}")
    else:
        print("flash_attn not available (slower attention)")

print("\nPackage setup complete")

Found offline packages at: /kaggle/input/sft-offline-packages (27 wheels)
Installing 27 wheels with --no-deps...
Install error: ERROR: Could not install packages due to an OSError: [Errno 30] Read-only file system: '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/fsspec-2026.2.0.dist-info/'


  OK: accelerate
  OK: aiohappyeyeballs
  OK: aiohttp
  OK: aiosignal
  OK: annotated_doc
  OK: attrs
  OK: click
  OK: datasets
  OK: dill
  OK: frozenlist
  FAIL: fsspec:  system: '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/fsspec-2026.2.0.dist-info/'


  OK: hf_xet
  OK: httpcore
  OK: httpx
  OK: markdown_it_py
  OK: mdurl
  OK: multidict
  OK: multiprocess
  OK: propcache
  OK: psutil
  OK: pygments
  OK: rich
  OK: shellingham
  OK: trl
  OK: typer
  OK: xxhash
  OK: yarl
  trl=0.29.1
  datasets=4.0.0
flash_attn=2.8.3

Package setup complete


In [2]:
import os
import sys
import stat
import shutil
import gc
import zipfile
import time
import json
import polars as pl
import pandas as pd
import torch
import torch.nn.functional as F
import kagglehub
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

print(f"torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    mem = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
    print(f"GPU memory: {mem / 1024**3:.1f} GB")

# --- Triton / Kaggle environment fixes ---
def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast:
        x = x.float()
    variance = x.pow(2).mean(-1, keepdim=True)
    x_normed = x * torch.rsqrt(variance + eps)
    out = x_normed * weight.float()
    if bias is not None:
        out = out + bias.float()
    if z is not None:
        out = out * F.silu(z.float())
    return out.to(dtype)

for name, mod in list(sys.modules.items()):
    if hasattr(mod, 'rmsnorm_fn'):
        mod.rmsnorm_fn = _pure_rmsnorm_fn

# Copy PTXAS binaries to writable temp
src = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell"
dst = "/tmp/ptxas-blackwell"
if os.path.exists(src):
    shutil.copy2(src, dst)
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    import triton.backends.nvidia as nv_backend
    src_bin = os.path.join(os.path.dirname(nv_backend.__file__), "bin")
    dst_bin = "/tmp/triton_nvidia_bin"
    shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
    for f in os.listdir(dst_bin):
        fp = os.path.join(dst_bin, f)
        if os.path.isfile(fp):
            os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    nv_backend.__file__ = os.path.join(dst_bin, "..", "__init__.py")
    os.environ["TRITON_PTXAS_PATH"] = dst
    print("✅ Triton patched")
else:
    print("⚠️ ptxas-blackwell not found, skipping Triton patch")

print("✅ Imports and environment ready")


/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/torch/compiler/__init__.py:148: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  return torch._dynamo.allow_in_graph(fn)


torch: 2.12.0.dev20260324+cu128, CUDA: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
GPU memory: 95.0 GB
✅ Triton patched
✅ Imports and environment ready


## Configuration

All hyperparameters are defined here. The `PROMPT_SUFFIX` is copied verbatim from the **official evaluation metric script** — do not modify it.


In [3]:
# =============================================
#  🔧 HYPERPARAMETERS — EDIT HERE
# =============================================

# --- Stage 1: Learn reasoning (thinking-only data) ---
STAGE1_N_SAMPLES = None    # None = all matching rows
STAGE1_THINKING_ONLY = True  # True = drop answer-only rows (keep ~11K with thinking)
STAGE1_DROP_LONG = True    # True = drop samples exceeding STAGE1_MAX_SEQ (zero truncation)
STAGE1_LR        = 2e-4
STAGE1_EPOCHS    = 1
STAGE1_MAX_SEQ   = 512     # Samples > 512 tokens will be dropped (not truncated)
STAGE1_BATCH     = 1       # Per-device batch size
STAGE1_GRAD_ACCUM = 4      # Effective batch = 1 × 4 = 4
STAGE1_PACKING   = True    # Safe with max_seq=512 + long samples dropped

# --- Stage 2: Format polish (answer-only, low LR) ---
STAGE2_ENABLED   = True
STAGE2_N_SAMPLES = 600
STAGE2_LR        = 4e-5    # 1/5 of Stage 1
STAGE2_EPOCHS    = 1
STAGE2_MAX_SEQ   = 512     # answer-only is short
STAGE2_BATCH     = 1
STAGE2_GRAD_ACCUM = 4
STAGE2_PACKING   = True

# --- LoRA ---
LORA_RANK        = 32
LORA_ALPHA       = 16
LORA_DROPOUT     = 0.05

# =============================================
#  🔴 OFFICIAL PROMPT SUFFIX — DO NOT MODIFY
# =============================================
# Source: competition_notebooks/nemotron-baseline-evaluation.ipynb
# Lines: user_content = item.prompt + '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'
PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

OUTPUT_DIR = "/kaggle/working/adapter"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Stage 1: n={STAGE1_N_SAMPLES}, thinking_only={STAGE1_THINKING_ONLY}, drop_long={STAGE1_DROP_LONG}")
print(f"Stage 1: LR={STAGE1_LR}, epochs={STAGE1_EPOCHS}, max_seq={STAGE1_MAX_SEQ}")
print(f"Stage 1: batch={STAGE1_BATCH}, grad_accum={STAGE1_GRAD_ACCUM}, packing={STAGE1_PACKING}")
print(f"Stage 2: enabled={STAGE2_ENABLED}, n={STAGE2_N_SAMPLES}, LR={STAGE2_LR}, packing={STAGE2_PACKING}")
print(f"LoRA: rank={LORA_RANK}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")
print(f"Prompt suffix: {repr(PROMPT_SUFFIX)}")

Stage 1: n=None, thinking_only=True, drop_long=True
Stage 1: LR=0.0002, epochs=1, max_seq=512
Stage 1: batch=1, grad_accum=4, packing=True
Stage 2: enabled=True, n=600, LR=4e-05, packing=True
LoRA: rank=32, alpha=16, dropout=0.05
Prompt suffix: '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'


## Data Loading & Statistics

In [4]:
MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
COMP_DATA = '/kaggle/input/nvidia-nemotron-3-reasoning-challenge'
COT_DATA = '/kaggle/input/prog-cot-training-data'

# Load the merged dataset
train_df = pl.read_csv(f'{COT_DATA}/sft_merged_v1.csv')

print(f"{'='*60}")
print(f"  Loaded: sft_merged_v1.csv — {len(train_df)} rows")
print(f"{'='*60}")

# Statistics
pdf = train_df.to_pandas()
has_thinking = pdf['thinking'].fillna('').str.strip().str.len() > 0
n_with = has_thinking.sum()
n_without = (~has_thinking).sum()
print(f"\n  With thinking: {n_with} ({n_with/len(pdf)*100:.1f}%)")
print(f"  Without thinking (answer-only): {n_without} ({n_without/len(pdf)*100:.1f}%)")

# Classify thinking types
short_mask = has_thinking & (pdf['thinking'].str.len() < 50)
long_mask = has_thinking & (pdf['thinking'].str.len() >= 50)
print(f"  - Compact rules (<50 chars): {short_mask.sum()}")
print(f"  - Full CoT (≥50 chars): {long_mask.sum()}")

# Show thinking length distribution
print(f"\n  Thinking length stats (non-empty):")
lengths = pdf.loc[has_thinking, 'thinking'].str.len()
print(f"    min={lengths.min()}, median={lengths.median():.0f}, mean={lengths.mean():.0f}, max={lengths.max()}")

# Check for any data issues
print(f"\n  --- Sanity checks ---")
print(f"  Empty prompt: {(pdf['prompt'].fillna('').str.len() == 0).sum()}")
print(f"  Empty answer: {(pdf['answer'].fillna('').astype(str).str.len() == 0).sum()}")
boxed_in_thinking = pdf.loc[has_thinking, 'thinking'].str.contains(r'\\boxed', regex=True, na=False).sum()
print(f"  \\boxed in thinking: {boxed_in_thinking}")
print(f"  Columns: {list(pdf.columns)}")


  Loaded: sft_merged_v1.csv — 15204 rows

  With thinking: 11204 (73.7%)
  Without thinking (answer-only): 4000 (26.3%)
  - Compact rules (<50 chars): 5069
  - Full CoT (≥50 chars): 6135

  Thinking length stats (non-empty):
    min=8, median=60, mean=169, max=1245

  --- Sanity checks ---
  Empty prompt: 0
  Empty answer: 0
  \boxed in thinking: 0
  Columns: ['id', 'prompt', 'answer', 'thinking']


## Prompt Format & Training Text

**2-Stage Format Design:**
- **Stage 1** (reasoning): NO boxed suffix, NO `\boxed{}` wrapping → pure reasoning training
- **Stage 2** (format): WITH boxed suffix, WITH `\boxed{}` wrapping → format alignment

The Stage 2 suffix must match the official evaluation metric script EXACTLY:
```python
user_content = item.prompt + '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'
```


In [5]:
import math

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# =============================================
#  Prompt suffix verification (for Stage 2)
# =============================================
print("=== PROMPT SUFFIX VERIFICATION ===")
print(f"Our PROMPT_SUFFIX: {repr(PROMPT_SUFFIX)}")

official_suffix = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'
assert PROMPT_SUFFIX == official_suffix, f"❌ MISMATCH!\nOurs:     {repr(PROMPT_SUFFIX)}\nOfficial: {repr(official_suffix)}"
print("✅ Prompt suffix matches official evaluation exactly")

# Show what official eval generates for inference
sample_prompt = "What is 2 + 2?"
official_inference_prompt = tokenizer.apply_chat_template(
    [{"role": "user", "content": sample_prompt + PROMPT_SUFFIX}],
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True,
)
print(f"\nOfficial inference prompt (enable_thinking=True):\n---\n{official_inference_prompt}\n---")

# =============================================
#  Helper: check if thinking is empty/missing
# =============================================
def _has_thinking(thinking):
    """Return True if thinking contains actual content (not NaN/None/empty)."""
    if thinking is None:
        return False
    if isinstance(thinking, float) and math.isnan(thinking):
        return False
    s = str(thinking).strip()
    return len(s) > 0 and s.lower() != 'nan'

# =============================================
#  Stage 1 builder: NO boxed, pure reasoning
# =============================================
def build_stage1_text(example):
    """Stage 1: Learn reasoning. NO boxed suffix, NO \\boxed{} wrapping."""
    prompt = example["prompt"]
    answer = str(example["answer"])
    thinking = example.get("thinking", None)
    
    if _has_thinking(thinking):
        text = (
            f"<|im_start|>user\n{prompt}<|im_end|>\n"
            f"<|im_start|>assistant\n<think>\n{str(thinking).strip()}\n</think>\n{answer}<|im_end|>"
        )
    else:
        text = (
            f"<|im_start|>user\n{prompt}<|im_end|>\n"
            f"<|im_start|>assistant\n<think></think>{answer}<|im_end|>"
        )
    return {"text": text}

# =============================================
#  Stage 2 builder: WITH boxed, format alignment
# =============================================
def build_stage2_text(example):
    """Stage 2: Format polish. WITH boxed suffix, WITH \\boxed{} wrapping.
    Answer-only, enable_thinking=False."""
    prompt = example["prompt"]
    answer = str(example["answer"])
    user_msg = prompt + PROMPT_SUFFIX
    
    assistant_msg = f"\\boxed{{{answer}}}"
    try:
        messages = [
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": assistant_msg},
        ]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False,
            add_generation_prompt=False,
            enable_thinking=False,
        )
    except Exception:
        text = (
            f"<|im_start|>user\n{user_msg}<|im_end|>\n"
            f"<|im_start|>assistant\n<think></think>\\boxed{{{answer}}}<|im_end|>"
        )
    return {"text": text}

# =============================================
#  Format verification with actual data
# =============================================
print("\n=== STAGE 1 FORMAT VERIFICATION ===")
sample_df = train_df.to_pandas()

# Test Stage 1 with a thinking row
think_rows = sample_df[sample_df['thinking'].apply(_has_thinking)]
if len(think_rows) > 0:
    row = think_rows.iloc[0].to_dict()
    result = build_stage1_text(row)
    text = result['text']
    print(f"\n--- STAGE 1: THINKING ROW (id={row['id']}) ---")
    print(text[:500])
    if len(text) > 500:
        print(f"... ({len(text)} chars total)")
    assert '<think>\n' in text, "Missing <think> tag"
    assert '\n</think>\n' in text, "Missing </think> tag"
    assert '\\boxed{' not in text, "❌ Stage 1 should NOT contain \\boxed{}!"
    assert 'boxed' not in text.split('<|im_start|>user')[1].split('<|im_end|>')[0], "❌ Stage 1 user msg should NOT mention boxed!"
    print("✅ Stage 1 thinking row: no boxed, has thinking")

# Test Stage 1 with an answer-only row
ao_rows = sample_df[~sample_df['thinking'].apply(_has_thinking)]
if len(ao_rows) > 0:
    row = ao_rows.iloc[0].to_dict()
    result = build_stage1_text(row)
    text = result['text']
    print(f"\n--- STAGE 1: ANSWER-ONLY ROW (id={row['id']}) ---")
    print(text[:500])
    assert '\\boxed{' not in text, "❌ Stage 1 should NOT contain \\boxed{}!"
    assert '<think></think>' in text, "Missing empty think tags"
    print("✅ Stage 1 answer-only row: no boxed, empty think")

print("\n=== STAGE 2 FORMAT VERIFICATION ===")
if len(ao_rows) > 0:
    row = ao_rows.iloc[0].to_dict()
    result = build_stage2_text(row)
    text = result['text']
    print(f"\n--- STAGE 2: ROW (id={row['id']}) ---")
    print(text[:500])
    assert PROMPT_SUFFIX.lstrip('\n') in text, "Missing prompt suffix"
    assert '\\boxed{' in text, "Missing \\boxed{}"
    print("✅ Stage 2 row: has boxed, has suffix")

print("\n✅ All format checks passed!")

=== PROMPT SUFFIX VERIFICATION ===
Our PROMPT_SUFFIX: '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'
✅ Prompt suffix matches official evaluation exactly

Official inference prompt (enable_thinking=True):
---
<|im_start|>system
<|im_end|>
<|im_start|>user
What is 2 + 2?
Please put your final answer inside `\boxed{}`. For example: `\boxed{your answer}`<|im_end|>
<|im_start|>assistant
<think>

---

=== STAGE 1 FORMAT VERIFICATION ===

--- STAGE 1: THINKING ROW (id=7e6b95cd) ---
<|im_start|>user
In Alice's Wonderland, secret encryption rules are used on text. Here are some examples:
bosxn nkagoznx wzexyrg -> mouse explores crystal
yhn rpwtnpy yszygn dztynx -> the ancient turtle writes
cptqhy utxwomnzx tpxtun argrwn -> knight discovers inside palace
Now, decrypt the following text: hryynz utxwomnzx yhn qogunp fooc<|im_end|>
<|im_start|>assistant
<think>
First, I'll find the substitution mapping by comparing the ciphertext and plaintext examples. Fro

In [6]:
# Convert dataset for Stage 1 (no boxed)
stage1_pdf = train_df.to_pandas()

# Filter: keep only rows with thinking if STAGE1_THINKING_ONLY
if STAGE1_THINKING_ONLY:
    before = len(stage1_pdf)
    stage1_pdf = stage1_pdf[stage1_pdf['thinking'].apply(_has_thinking)].reset_index(drop=True)
    print(f"Stage 1 thinking-only filter: {before} → {len(stage1_pdf)} rows (dropped {before - len(stage1_pdf)} answer-only)")

if STAGE1_N_SAMPLES is not None:
    stage1_pdf = stage1_pdf.sample(n=min(STAGE1_N_SAMPLES, len(stage1_pdf)), random_state=42)
    print(f"Stage 1 subsampled: {len(stage1_pdf)} rows")

hf_dataset = Dataset.from_pandas(stage1_pdf)
hf_dataset = hf_dataset.map(
    build_stage1_text,
    remove_columns=hf_dataset.column_names,
)
print(f"Stage 1 dataset formatted: {len(hf_dataset)} rows (no boxed)")

# Token length analysis
print("\n=== TOKEN LENGTH ANALYSIS ===")
import numpy as np

token_lengths = []
for i in range(len(hf_dataset)):
    toks = tokenizer(hf_dataset[i]['text'], add_special_tokens=False)
    token_lengths.append(len(toks['input_ids']))

tl = np.array(token_lengths)
print(f"  Total samples: {len(tl)}")
print(f"  Min tokens:    {tl.min()}")
print(f"  Median tokens: {np.median(tl):.0f}")
print(f"  Mean tokens:   {tl.mean():.0f}")
print(f"  P95 tokens:    {np.percentile(tl, 95):.0f}")
print(f"  P99 tokens:    {np.percentile(tl, 99):.0f}")
print(f"  Max tokens:    {tl.max()}")

# Drop samples exceeding max_seq if enabled
if STAGE1_DROP_LONG:
    keep_mask = tl <= STAGE1_MAX_SEQ
    n_drop = (~keep_mask).sum()
    print(f"\n  Dropping {n_drop} samples > {STAGE1_MAX_SEQ} tokens ({n_drop/len(tl)*100:.1f}%)")
    keep_indices = np.where(keep_mask)[0].tolist()
    hf_dataset = hf_dataset.select(keep_indices)
    tl = tl[keep_mask]
    print(f"  Remaining: {len(hf_dataset)} samples (max={tl.max()} tokens, zero truncation)")
else:
    truncated = (tl > STAGE1_MAX_SEQ).sum()
    print(f"\n  Truncated at {STAGE1_MAX_SEQ}: {truncated} ({truncated/len(tl)*100:.1f}%)")

# Show 3 samples: short, medium, long
sorted_idx = np.argsort(tl)
for label, idx in [("SHORTEST", sorted_idx[0]), ("MEDIAN", sorted_idx[len(sorted_idx)//2]), ("LONGEST", sorted_idx[-1])]:
    print(f"\n--- {label} ({tl[idx]} tokens) ---")
    print(hf_dataset[int(idx)]['text'][:400])
    if len(hf_dataset[int(idx)]['text']) > 400:
        print(f"... [truncated, {len(hf_dataset[int(idx)]['text'])} chars]")

Stage 1 thinking-only filter: 15204 → 11204 rows (dropped 4000 answer-only)


Map:   0%|          | 0/11204 [00:00<?, ? examples/s]

Stage 1 dataset formatted: 11204 rows (no boxed)

=== TOKEN LENGTH ANALYSIS ===
  Total samples: 11204
  Min tokens:    73
  Median tokens: 193
  Mean tokens:   240
  P95 tokens:    530
  P99 tokens:    633
  Max tokens:    906

  Dropping 689 samples > 512 tokens (6.1%)
  Remaining: 10515 samples (max=512 tokens, zero truncation)

--- SHORTEST (73 tokens) ---
<|im_start|>user
In Alice's Wonderland, numbers are secretly converted into a different numeral system. Some examples are given below:
36 -> XXXVI
9 -> IX
5 -> V
Now, write the number 4 in the Wonderland numeral system.<|im_end|>
<|im_start|>assistant
<think>
[ROMAN_GREEDY]
</think>
IV<|im_end|>

--- MEDIAN (183 tokens) ---
<|im_start|>user
In Alice's Wonderland, secret encryption rules are used on text. Here are some examples:
hdiirj saas jphmzqp kmztjdrt -> rabbit sees through mountain
hdiirj ndjvpas sjmhb -> rabbit watches story
yhdqmt yhadks ztyah nmtyahcdty -> dragon dreams under wonderland
kmzsa saas ozggca -> mouse sees pu

## Model Loading & LoRA Setup

In [7]:
# Mock cutlass/mamba3 to prevent import errors
from unittest.mock import MagicMock

_mock_modules = [
    "cutlass", "cutlass.cute", "cutlass.utils",
    "mamba_ssm.ops.cute", "mamba_ssm.ops.cute.mamba3",
    "mamba_ssm.ops.cute.mamba3.mamba3_step_fn",
    "mamba_ssm.ops.tilelang", "mamba_ssm.ops.tilelang.mamba3",
    "mamba_ssm.ops.tilelang.mamba3.mamba3_mimo",
]
for mod_name in _mock_modules:
    if mod_name not in sys.modules:
        sys.modules[mod_name] = MagicMock()
print(f"Mock modules injected: {len(_mock_modules)}")

# Load model
t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    trust_remote_code=True,
    dtype=torch.bfloat16,
)
print(f"Model loaded in {time.time()-t0:.1f}s. Vocab size: {len(tokenizer)}")

# Patch: force slow path to bypass broken CUDA kernels
for name, mod in sys.modules.items():
    if "modeling_nemotron_h" in name:
        mod.is_fast_path_available = False
        print(f"Patched {name}: is_fast_path_available = False")

# Setup LoRA
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules="all-linear",
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Log model architecture summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal params: {total_params/1e9:.2f}B")
print(f"Trainable params: {trainable_params/1e6:.1f}M ({trainable_params/total_params*100:.2f}%)")


Mock modules injected: 9


Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

Model loaded in 538.3s. Vocab size: 131072
Patched transformers_modules._1.modeling_nemotron_h: is_fast_path_available = False


/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:122: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:195: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_scaling_utils.py:90: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_linear.py:60: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_

trainable params: 883,873,792 || all params: 32,461,811,136 || trainable%: 2.7228

Total params: 32.46B
Trainable params: 883.9M (2.72%)


## Stage 1: Learn Reasoning

Train on all 15K samples (compact rules + full CoT + answer-only). This teaches the model to:
1. Recognize different problem types
2. Apply reasoning strategies within `<think>` tags  
3. Output answers in `\boxed{}` format


In [8]:
import triton.backends.nvidia.compiler as nv_compiler
os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = "/tmp/ptxas-blackwell"
nv_compiler.get_ptxas_version = lambda arch: "12.0"

eff_batch = STAGE1_BATCH * STAGE1_GRAD_ACCUM
total_steps = (len(hf_dataset) // eff_batch) * STAGE1_EPOCHS
print(f"{'='*60}")
print(f"  STAGE 1: Reasoning Training")
print(f"  Samples: {len(hf_dataset)}, LR: {STAGE1_LR}, Epochs: {STAGE1_EPOCHS}")
print(f"  Max Seq: {STAGE1_MAX_SEQ}, Batch: {STAGE1_BATCH}, Grad Accum: {STAGE1_GRAD_ACCUM}")
print(f"  Packing: {STAGE1_PACKING}, Effective batch: {eff_batch}")
print(f"  Estimated steps: ~{total_steps}")
print(f"{'='*60}")

stage1_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=STAGE1_BATCH,
    gradient_accumulation_steps=STAGE1_GRAD_ACCUM,
    num_train_epochs=STAGE1_EPOCHS,
    learning_rate=STAGE1_LR,
    logging_steps=10,
    bf16=True,
    max_grad_norm=1.0,
    optim="adamw_torch",
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    save_strategy="no",
    report_to="none",
    dataset_text_field="text",
    max_length=STAGE1_MAX_SEQ,
    packing=STAGE1_PACKING,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": True},
)

stage1_trainer = SFTTrainer(
    model=model,
    train_dataset=hf_dataset,
    processing_class=tokenizer,
    args=stage1_args,
)

t0 = time.time()
stage1_result = stage1_trainer.train()
stage1_time = time.time() - t0

print(f"\n{'='*60}")
print(f"  STAGE 1 COMPLETE")
print(f"  Time: {stage1_time/60:.1f} min")
print(f"  Final loss: {stage1_result.training_loss:.4f}")
print(f"{'='*60}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


  STAGE 1: Reasoning Training
  Samples: 10515, LR: 0.0002, Epochs: 1
  Max Seq: 512, Batch: 1, Grad Accum: 4
  Packing: True, Effective batch: 4
  Estimated steps: ~2628


Adding EOS to train dataset:   0%|          | 0/10515 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/10515 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/10515 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 11, 'pad_token_id': 11}.


Step,Training Loss
10,8.892204
20,8.886447
30,7.903700
40,6.418280
50,5.801858
60,4.381690
70,3.393048
80,4.008438
90,3.590364
100,3.303923



  STAGE 1 COMPLETE
  Time: 274.4 min
  Final loss: 3.1045


## Stage 2: Format Polish (Optional)

Light fine-tuning with answer-only data at low LR. Reinforces `\boxed{}` output format without disrupting learned reasoning.

Set `STAGE2_ENABLED = False` above to skip this stage.


In [9]:
if STAGE2_ENABLED:
    print(f"{'='*60}")
    print(f"  STAGE 2: Format Polish")
    print(f"{'='*60}")
    
    # Prepare Stage 2 dataset from answer-only rows
    full_df = train_df.to_pandas()
    ao_mask = full_df['thinking'].fillna('').str.strip().str.len() == 0
    answer_only_df = full_df[ao_mask]
    
    n_sample = min(STAGE2_N_SAMPLES, len(answer_only_df))
    stage2_df = answer_only_df.sample(n=n_sample, random_state=42)
    
    eff_batch2 = STAGE2_BATCH * STAGE2_GRAD_ACCUM
    total_steps2 = (n_sample // eff_batch2) * STAGE2_EPOCHS
    print(f"  Answer-only pool: {len(answer_only_df)}")
    print(f"  Sampled for Stage 2: {n_sample}")
    print(f"  LR: {STAGE2_LR}, Epochs: {STAGE2_EPOCHS}, Max Seq: {STAGE2_MAX_SEQ}")
    print(f"  Batch: {STAGE2_BATCH}, Grad Accum: {STAGE2_GRAD_ACCUM}, Packing: {STAGE2_PACKING}")
    print(f"  Estimated steps: ~{total_steps2}")
    
    stage2_dataset = Dataset.from_pandas(stage2_df)
    stage2_dataset = stage2_dataset.map(
        build_stage2_text,
        remove_columns=stage2_dataset.column_names,
    )
    
    # Show a Stage 2 sample
    print(f"\n--- Stage 2 sample ---")
    print(stage2_dataset[0]['text'][:300])
    
    stage2_args = SFTConfig(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=STAGE2_BATCH,
        gradient_accumulation_steps=STAGE2_GRAD_ACCUM,
        num_train_epochs=STAGE2_EPOCHS,
        learning_rate=STAGE2_LR,
        logging_steps=5,
        bf16=True,
        max_grad_norm=1.0,
        optim="adamw_torch",
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        save_strategy="no",
        report_to="none",
        dataset_text_field="text",
        max_length=STAGE2_MAX_SEQ,
        packing=STAGE2_PACKING,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": True},
    )
    
    stage2_trainer = SFTTrainer(
        model=model,  # Same model object — continues from Stage 1
        train_dataset=stage2_dataset,
        processing_class=tokenizer,
        args=stage2_args,
    )
    
    t0 = time.time()
    stage2_result = stage2_trainer.train()
    stage2_time = time.time() - t0
    
    print(f"\n{'='*60}")
    print(f"  STAGE 2 COMPLETE")
    print(f"  Time: {stage2_time/60:.1f} min")
    print(f"  Final loss: {stage2_result.training_loss:.4f}")
    print(f"{'='*60}")
else:
    print("Stage 2 SKIPPED (STAGE2_ENABLED=False)")

  STAGE 2: Format Polish
  Answer-only pool: 4000
  Sampled for Stage 2: 600
  LR: 4e-05, Epochs: 1, Max Seq: 512
  Batch: 1, Grad Accum: 4, Packing: True
  Estimated steps: ~150


Map:   0%|          | 0/600 [00:00<?, ? examples/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.



--- Stage 2 sample ---
<|im_start|>system
<|im_end|>
<|im_start|>user
In Alice's Wonderland, the gravitational constant has been secretly changed. Here are some example observations:
For t = 3.66s, distance = 85.11 m
For t = 4.62s, distance = 135.61 m
For t = 3.45s, distance = 75.62 m
For t = 1.23s, distance = 9.61 m
Now,


Adding EOS to train dataset:   0%|          | 0/600 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/600 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/600 [00:00<?, ? examples/s]

Step,Training Loss
5,6.327249
10,4.305812
15,3.550116
20,3.065231
25,2.876831
30,2.312178
35,2.484787
40,2.458467
45,2.456279
50,2.668528



  STAGE 2 COMPLETE
  Time: 12.9 min
  Final loss: 3.1856


## Save & Package Submission

In [10]:
# Save adapter weights and config
model.save_pretrained(OUTPUT_DIR)
print(f"Adapter saved to {OUTPUT_DIR}")
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f:40s} {size/1024:.1f} KB")

# Package into submission.zip
zip_path = "/kaggle/working/submission.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(OUTPUT_DIR):
        fpath = os.path.join(OUTPUT_DIR, fname)
        if os.path.isfile(fpath):
            zf.write(fpath, fname)  # files at zip root

print(f"\nCreated {zip_path} ({os.path.getsize(zip_path)/1024/1024:.1f} MB)")

# Verify submission
with zipfile.ZipFile(zip_path, 'r') as zf:
    names = zf.namelist()
    print(f"Contents: {names}")
    assert "adapter_config.json" in names, "❌ Missing adapter_config.json!"
    assert "adapter_model.safetensors" in names, "❌ Missing adapter_model.safetensors!"
    
    # Print adapter config
    with zf.open("adapter_config.json") as cf:
        config = json.loads(cf.read())
        print(f"\nAdapter config:")
        print(f"  r (rank):     {config.get('r', '?')}")
        print(f"  lora_alpha:   {config.get('lora_alpha', '?')}")
        print(f"  target_modules: {config.get('target_modules', '?')[:80]}")

print("\n✅ submission.zip is valid and ready to submit!")


Adapter saved to /kaggle/working/adapter
  README.md                                5.2 KB
  adapter_config.json                      1.1 KB
  adapter_model.safetensors                3454393.7 KB

Created /kaggle/working/submission.zip (3101.4 MB)
Contents: ['adapter_model.safetensors', 'adapter_config.json', 'README.md']

Adapter config:
  r (rank):     32
  lora_alpha:   16
  target_modules: ['in_proj', 'k_proj', 'up_proj', 'v_proj', 'down_proj', 'q_proj', 'o_proj', 'out_proj']

✅ submission.zip is valid and ready to submit!


In [11]:
print("=" * 60)
print("  TRAINING SUMMARY")
print("=" * 60)
print(f"  Stage 1: {len(hf_dataset)} samples, loss={stage1_result.training_loss:.4f}, time={stage1_time/60:.1f}min")
if STAGE2_ENABLED:
    print(f"  Stage 2: {n_sample} samples, loss={stage2_result.training_loss:.4f}, time={stage2_time/60:.1f}min")
    print(f"  Total time: {(stage1_time + stage2_time)/60:.1f} min")
else:
    print(f"  Stage 2: SKIPPED")
    print(f"  Total time: {stage1_time/60:.1f} min")
print(f"  LoRA rank: {LORA_RANK}, alpha: {LORA_ALPHA}")
print(f"  Prompt suffix verified: ✅")
print(f"  Output: /kaggle/working/submission.zip")
print("=" * 60)


  TRAINING SUMMARY
  Stage 1: 10515 samples, loss=3.1045, time=274.4min
  Stage 2: 600 samples, loss=3.1856, time=12.9min
  Total time: 287.3 min
  LoRA rank: 32, alpha: 16
  Prompt suffix verified: ✅
  Output: /kaggle/working/submission.zip
